# Assignment 2
This fully connected neural network predicts the median house value in California districts.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from keras.models import Sequential
from keras.layers import Input, Dense
from keras.datasets import california_housing

(inputs, targets), (x_test, y_test) = california_housing.load_data(version='large', 
                                        test_split=0.2)
print(inputs.shape, targets.shape)
print(x_test.shape, y_test.shape)

I0000 00:00:1774868094.408168   53077 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


(16512, 8) (16512,)
(4128, 8) (4128,)


In [2]:
import math
# Get 80% of all inputs, since targets is the same size this also applies to it
split = math.floor(inputs.shape[0] * 0.8)
x_train = inputs[:split]
y_train = targets[:split]
x_val = inputs[split:]
y_val = targets[split:]

In [3]:
mean = x_train.mean(axis=0) # mean and standard deviation computed from training set only
std = x_train.std(axis=0)
x_train -= mean
x_train /= std
x_test -= mean
x_test /= std
x_val -= mean
x_val /= std
y_train /= 1e5
y_test /= 1e5
y_val /= 1e5

In [4]:
# define the model architecture
model = Sequential([
    Input(shape=(8,)),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(256, activation='relu'),
    Dense(1)
    ])

# configure the learning algorithm
model.compile(optimizer='adam',loss='mse',metrics=['mae'])

I0000 00:00:1774868098.530214   53077 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1767 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [ ]:
history = model.fit(x_train, y_train, epochs=200, batch_size=64, validation_data=(x_val, y_val), verbose=0)

I0000 00:00:1774868101.571776   53135 service.cc:153] XLA service 0x703aa00320b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774868101.571842   53135 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6 (Driver: 12.8.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1774868101.614107   53135 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1774868101.858163   53135 cuda_dnn.cc:461] Loaded cuDNN version 92000
I0000 00:00:1774868101.873667   53135 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1568__.10
I0000 00:00:1774868104.118944   53205 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_28', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1774868107.258763   53135 device_compiler.h:208] Compiled c

In [ ]:
plt.plot(history.history['loss'][10:], label='train loss')
plt.plot(history.history['val_loss'][10:], label='val loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')

In [ ]:
plt.plot(history.history['mae'][10:], label='train MAE')
plt.plot(history.history['val_mae'][10:], label='val MAE')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Mean absolute error (in 100000 $)')

In [ ]:
# First trying early stopping
# define the model architecture
model = Sequential([
    Input(shape=(8,)),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(256, activation='relu'),
    Dense(1)
    ])

# configure the learning algorithm
model.compile(optimizer='adam',loss='mse',metrics=['mae'])

In [ ]:
from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=10)
history = model.fit(x_train, y_train, epochs=200, batch_size=64, validation_data=(x_val, y_val), callbacks=[early_stopping])

In [ ]:
plt.plot(history.history['mae'][10:], label='train MAE')
plt.plot(history.history['val_mae'][10:], label='val MAE')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Mean absolute error (in 100000 $)')

In [ ]:
# Regularization
from keras.regularizers import l2

model = Sequential([
    Input(shape=(8,)),
    Dense(512, activation='relu', kernel_regularizer=l2(0.01)),
    Dense(256, activation='relu', kernel_regularizer=l2(0.01)),
    Dense(256, activation='relu', kernel_regularizer=l2(0.01)),
    Dense(1)
])
model.compile(optimizer='adam',loss='mse',metrics=['mae'])

In [ ]:
history = model.fit(x_train, y_train, epochs=200, batch_size=64, validation_data=(x_val, y_val), verbose=0)

In [ ]:
plt.plot(history.history['mae'][10:], label='train MAE')
plt.plot(history.history['val_mae'][10:], label='val MAE')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Mean absolute error (in 100000 $)')

In [ ]:
from keras.layers import Dropout
model = Sequential([
    Input(shape=(8,)),
    Dense(512, activation='relu'),
    Dropout(0.2),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(1)
    ])

# configure the learning algorithm
model.compile(optimizer='adam',loss='mse',metrics=['mae'])

In [ ]:
history = model.fit(x_train, y_train, epochs=200, batch_size=64, validation_data=(x_val, y_val), verbose=0)

In [ ]:
plt.plot(history.history['mae'][10:], label='train MAE')
plt.plot(history.history['val_mae'][10:], label='val MAE')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Mean absolute error (in 100000 $)')

In [ ]:
from keras.layers import BatchNormalization
model = Sequential([
    Input(shape=(8,)),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dense(1)
    ])

# configure the learning algorithm
model.compile(optimizer='adam',loss='mse',metrics=['mae'])

In [ ]:
history = model.fit(x_train, y_train, epochs=200, batch_size=64, validation_data=(x_val, y_val), verbose=0)

In [ ]:
plt.plot(history.history['mae'][10:], label='train MAE')
plt.plot(history.history['val_mae'][10:], label='val MAE')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Mean absolute error (in 100000 $)')